### Model 1 (continued): Quantile Regression + Confidence Flag + Saved Outputs

This picks up exactly where `02_model_1_training.ipynb` left off. It assumes you already have
`../data/processed/model1_features.csv` (built in notebook 01).

What this notebook adds on top of your existing RF/XGBoost point-prediction baseline:
1. A **cold-start / low-confidence flag** per invoice (customers with too little history).
2. **Quantile Regression Forest** for P10 / P50 / P90 days-to-payment, from a single RF model
   (not three separately-trained models — RF outperformed XGBoost on point accuracy in notebook
   02, so we get quantiles as a bonus from that same model via per-tree percentiles).
3. **Guaranteed non-crossing quantiles** (P10 ≤ P50 ≤ P90 falls out of `np.percentile` for free
   with this method — no separate fix step needed, unlike training 3 independent models).
4. **Proper evaluation for quantile models**: pinball loss (not MAE) + calibration coverage
   (does the true value fall inside [P10, P90] roughly 80% of the time, as it should?).
5. **Saved model artifacts** (preprocessor + 1 RF model) via joblib, so Model 2 (Monte
   Carlo) and your FastAPI serving layer can load them without retraining.
6. **The actual spec output**: a per-invoice table with predicted payment *date range*
   (P10/P50/P90) and a confidence flag — this is what Model 2 and the SHAP layer consume.

**Why quantile regression, concretely:** the archetype breakdown from notebook 02 showed
`erratic_payer` (72 rows) at ~21.6 MAE vs. 3-6 for most other archetypes — one archetype
responsible for ~29% of total validation error despite being ~10% of the data. That's not a
fixable bug, it's genuine unpredictability. A point model has to guess one number anyway; a
quantile model can instead say "could be anywhere from 25 to 100 days" for that customer and be
right about being uncertain, while staying tight and accurate for `prompt_payer`/`average_payer`.
Section 8 below checks this directly.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 120)

## 1. Reload data and rebuild the same train/val/test split

In [2]:
df = pd.read_csv("../data/processed/model1_features.csv")
df["issue_date"] = pd.to_datetime(df["issue_date"])

feature_columns = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_invoice_count",
    "customer_payment_std",
    "payment_behavior_trend",
    "previous_payment_days",
    "invoice_amount",
    "payment_term_days",
    "sector"
]

numeric_features = [c for c in feature_columns if c != "sector"]
categorical_features = ["sector"]
target = "days_to_payment"

# id_columns and archetype are carried through model1_features.csv for traceability/diagnostics
# ONLY - never put them in feature_columns, archetype in particular is the ground-truth
# generating label and would be pure leakage if used as a model input.
id_columns = ["invoice_id", "cust_number"]
diagnostic_columns = ["customer_archetype_TRUE_LABEL"]

# Same cutoffs as notebook 02 - keep these identical across notebooks so results are comparable
train_df = df[df["issue_date"] <= "2025-10-05"].copy()
validation_df = df[(df["issue_date"] >= "2025-10-06") & (df["issue_date"] <= "2026-02-14")].copy()
test_df = df[df["issue_date"] >= "2026-02-15"].copy()

train_df.shape, validation_df.shape, test_df.shape

((3473, 14), (745, 14), (740, 14))

## 2. Cold-start / low-confidence flag

Per the spec: *"Cold-start customers (no history): fall back to sector-average priors, and flag
the prediction as low-confidence."*

`customer_invoice_count` already tells you how many prior invoices that customer had **before**
this one (it's 0 for a brand new customer, thanks to the `cumcount()` you did in notebook 01).
We flag anything with fewer than 3 prior invoices as low-confidence, since the historical
averages/std are noisy or entirely NaN before that point (you saw this in `.isna().sum()` in
notebook 01/02 — 180 NaNs on `customer_avg_payment_days`, which is exactly the "first invoice per
customer" count of 180 unique customers).

In [3]:
MIN_HISTORY_FOR_CONFIDENCE = 3

def add_confidence_flag(frame):
    frame = frame.copy()
    frame["is_cold_start"] = frame["customer_invoice_count"] < MIN_HISTORY_FOR_CONFIDENCE
    return frame

train_df = add_confidence_flag(train_df)
validation_df = add_confidence_flag(validation_df)
test_df = add_confidence_flag(test_df)

validation_df["is_cold_start"].value_counts()

is_cold_start
False    734
True      11
Name: count, dtype: int64

## 3. Sector-average prior for cold-start rows

Right now cold-start rows get the *global* median imputed for their missing historical features
(via the `SimpleImputer(strategy="median")` in your existing pipeline). The spec asks for a
**sector-average prior** instead — a customer in Textiles with no history should be compared to
other Textiles customers, not to the whole dataset. We compute this from `train_df` only (never
from validation/test, to avoid leakage) and use it to fill the NaNs before the median imputer
even sees them.

In [4]:
historical_features = [
    "customer_avg_payment_days",
    "customer_recent_avg_payment_days",
    "customer_payment_std",
    "payment_behavior_trend",
    "previous_payment_days",
]

# Sector-level average days-to-payment, computed on TRAIN ONLY
sector_priors = train_df.groupby("sector")["days_to_payment"].mean()
global_prior = train_df["days_to_payment"].mean()

def apply_sector_prior(frame):
    frame = frame.copy()
    sector_fill = frame["sector"].map(sector_priors).fillna(global_prior)
    # a brand-new customer has no "previous invoice" either - same sector-average fallback
    for col in ["customer_avg_payment_days", "customer_recent_avg_payment_days", "previous_payment_days"]:
        frame[col] = frame[col].fillna(sector_fill)
    # std/trend have no sensible sector prior - 0 variance / 0 trend is the safe neutral default
    frame["customer_payment_std"] = frame["customer_payment_std"].fillna(0)
    frame["payment_behavior_trend"] = frame["payment_behavior_trend"].fillna(0)
    return frame

train_df = apply_sector_prior(train_df)
validation_df = apply_sector_prior(validation_df)
test_df = apply_sector_prior(test_df)

train_df[historical_features].isna().sum()

customer_avg_payment_days           0
customer_recent_avg_payment_days    0
customer_payment_std                0
payment_behavior_trend              0
previous_payment_days               0
dtype: int64

## 4. Preprocessing pipeline

Same shape as notebook 02, but the numeric imputer now only needs to catch stragglers (there
shouldn't be any left after the sector-prior fill above, but keeping it is cheap insurance for
production data that might be messier than this synthetic set).

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

X_train, y_train = train_df[feature_columns], train_df[target]
X_val, y_val = validation_df[feature_columns], validation_df[target]
X_test, y_test = test_df[feature_columns], test_df[target]

preprocessor.fit(X_train)
X_train_p = preprocessor.transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

X_train_p.shape, X_val_p.shape, X_test_p.shape

((3473, 18), (745, 18), (740, 18))

## 5. Quantile Random Forest: P10 / P50 / P90 from a single model

Per your comparison in notebook 02, RF beat XGBoost on point accuracy (7.27 vs. 7.89 MAE), so
we're standardizing on RF rather than training three separate XGBoost models.

**How you get quantiles out of a model that only has `.predict()`:** a Random Forest is really
just 200 individual trees averaged together. `.predict()` throws away information by only
returning that average. Instead, we ask *each of the 200 trees* for its own prediction on a
sample, then take the 10th/50th/90th percentile *across those 200 numbers*. Customers with
stable, predictable behavior will have most trees agreeing (narrow spread → tight interval).
Customers like `erratic_payer` will have trees disagreeing a lot (wide spread → wide interval) —
which is exactly the honesty you want, and it falls out of a model you already trained, no extra
training run needed. This is the standard "Quantile Regression Forest" technique (Meinshausen,
2006), approximated here via per-tree predictions rather than full per-leaf sample pooling — a
common simplification which is more than good enough for this use case.

In [6]:
from sklearn.ensemble import RandomForestRegressor

rf_quantile_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
rf_quantile_model.fit(X_train_p, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [7]:
QUANTILES = {"p10": 0.10, "p50": 0.50, "p90": 0.90}

def rf_quantile_predict(rf_model, X, quantiles=QUANTILES):
    """Per-tree prediction spread -> percentiles across trees. X must already be preprocessed."""
    all_tree_preds = np.array([tree.predict(X) for tree in rf_model.estimators_])  # (n_trees, n_samples)
    percentiles = np.percentile(all_tree_preds, [q * 100 for q in quantiles.values()], axis=0)  # (3, n_samples)
    return pd.DataFrame(percentiles.T, columns=list(quantiles.keys()))

## 6. Predict on validation

Note: unlike training 3 separate models, `np.percentile` guarantees P10 ≤ P50 ≤ P90 by
construction (percentiles of the same underlying array can't cross), so there's no separate
crossing-fix step needed here — that was only a concern with the 3-independent-models approach.

In [8]:
val_preds = rf_quantile_predict(rf_quantile_model, X_val_p)
val_preds.head()

,p10,p50,p90
0,59.0,67.0,77.0
1,64.0,71.0,78.0
2,59.9,69.0,75.0
3,85.0,91.0,96.0
4,89.0,90.0,101.0


## 7. Evaluate the quantile model properly

MAE isn't the right metric for a quantile model. Two things matter instead:

- **Pinball loss** (a.k.a. quantile loss) on each quantile — the correct loss function for
  quantile regression, penalizing over- and under-prediction asymmetrically depending on `alpha`.
- **Calibration / coverage** — if P10–P90 is a genuine 80% interval, then roughly 80% of actual
  `days_to_payment` values in validation should fall inside it. If coverage is way off (e.g. 95%
  or 60%), the interval is miscalibrated and worth mentioning as a known limitation in the demo
  rather than presenting it as ground truth. This is the specific thing to watch for with the
  tree-percentile approximation — if coverage is poor, consider widening to P5/P95 or installing
  the `quantile-forest` package for the more rigorous per-leaf-sample version.

In [9]:
from sklearn.metrics import mean_pinball_loss

for name, alpha in QUANTILES.items():
    loss = mean_pinball_loss(y_val, val_preds[name], alpha=alpha)
    print(f"{name} (alpha={alpha}): pinball loss = {loss:.3f}")

p10 (alpha=0.1): pinball loss = 1.344
p50 (alpha=0.5): pinball loss = 3.195
p90 (alpha=0.9): pinball loss = 2.064


In [10]:
coverage = ((y_val.values >= val_preds["p10"]) & (y_val.values <= val_preds["p90"])).mean()
print(f"P10-P90 coverage on validation: {coverage:.1%}  (target: ~80%)")

# P50 should behave roughly like a point estimate - sanity check against your RF/XGB MAE from notebook 02
from sklearn.metrics import mean_absolute_error
p50_mae = mean_absolute_error(y_val, val_preds["p50"])
print(f"P50 MAE: {p50_mae:.2f} days  (compare to RF 7.27 / point-XGB 7.89 from notebook 02, post previous_payment_days fix)")

P10-P90 coverage on validation: 82.0%  (target: ~80%)
P50 MAE: 6.39 days  (compare to RF 7.27 / point-XGB 7.89 from notebook 02, post previous_payment_days fix)


## 8. Cold-start AND erratic customers should show wider intervals

Two sanity checks, both demo-relevant:

1. **Cold-start invoices** should have a wider P10-P90 spread than established customers, since
   the model has less to go on. If this comes out flat, `is_cold_start` isn't driving different
   behavior and is worth revisiting.
2. **`erratic_payer` archetype should have the widest bands of all archetypes.** You found earlier
   that `erratic_payer` alone has a point-model MAE of ~21.6 days (vs. ~3-6 for most other
   archetypes) - almost 3x the second worst. A point estimate can't represent that honestly; a
   correctly-behaving quantile model should instead give erratic customers a wide interval and
   `prompt_payer`/`average_payer` a tight one. `customer_archetype_TRUE_LABEL` is never a model
   input (it's the generating ground truth - using it would be leakage) - it's only used here,
   after the fact, to check whether the model's *uncertainty* lines up with reality.

In [11]:
check = validation_df[["is_cold_start"]].copy()
check["interval_width"] = val_preds["p90"].values - val_preds["p10"].values
check.groupby("is_cold_start")["interval_width"].mean()

is_cold_start
False    24.947684
True     30.663636
Name: interval_width, dtype: float64

In [12]:
if "customer_archetype_TRUE_LABEL" in validation_df.columns:
    archetype_check = validation_df[["customer_archetype_TRUE_LABEL"]].copy()
    archetype_check["interval_width"] = val_preds["p90"].values - val_preds["p10"].values
    archetype_check["actual_days"] = y_val.values
    archetype_check["p50"] = val_preds["p50"].values
    archetype_check["point_abs_error"] = (archetype_check["actual_days"] - archetype_check["p50"]).abs()
    archetype_check["in_interval"] = (
        (archetype_check["actual_days"] >= val_preds["p10"].values) &
        (archetype_check["actual_days"] <= val_preds["p90"].values)
    )

    summary = archetype_check.groupby("customer_archetype_TRUE_LABEL").agg(
        mean_interval_width=("interval_width", "mean"),
        p50_mae=("point_abs_error", "mean"),
        coverage=("in_interval", "mean"),
        count=("actual_days", "size"),
    ).sort_values("mean_interval_width", ascending=False)

    print(summary)
    print()
    print("Expect erratic_payer at the top (widest interval) with coverage still near 80%,")
    print("even though its p50_mae is high - that's the model being honestly uncertain")
    print("instead of confidently wrong.")
else:
    print("customer_archetype_TRUE_LABEL not found - make sure notebook 01's model1_features.csv")
    print("export includes diagnostic_columns as described.")

                               mean_interval_width    p50_mae  coverage  count
customer_archetype_TRUE_LABEL                                                 
erratic_payer                            74.265278  16.354167  0.791667     72
deteriorating_payer                      30.885937  11.593750  0.671875     64
cold_start                               29.200000   9.375000  0.500000     12
improving_payer                          28.372857   6.957143  0.814286     70
seasonal_payer                           22.722727   5.585227  0.840909     88
average_payer                            20.398544   4.643204  0.854369    206
chronic_late_payer                       17.078947   5.201754  0.894737     57
prompt_payer                             10.303409   2.823864  0.835227    176

Expect erratic_payer at the top (widest interval) with coverage still near 80%,
even though its p50_mae is high - that's the model being honestly uncertain
instead of confidently wrong.


## 9. Save everything (preprocessor + the one RF model) for reuse

This is what Model 2 (Monte Carlo) and your FastAPI serving layer will load — don't retrain
inside the API, load these artifacts at startup instead. Only one model file now, since
`rf_quantile_predict` derives all three quantiles from it at inference time.

In [13]:
import joblib

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

joblib.dump(preprocessor, model_dir / "model1_preprocessor.joblib")
joblib.dump(rf_quantile_model, model_dir / "model1_rf_quantile.joblib")
joblib.dump(sector_priors, model_dir / "model1_sector_priors.joblib")
joblib.dump(global_prior, model_dir / "model1_global_prior.joblib")

list(model_dir.glob("model1_*"))

[WindowsPath('../models/model1_global_prior.joblib'),
 WindowsPath('../models/model1_preprocessor.joblib'),
 WindowsPath('../models/model1_rf_quantile.joblib'),
 WindowsPath('../models/model1_sector_priors.joblib')]

## 10. Produce the actual spec output: per-invoice payment date range + confidence flag

This is the deliverable your architecture doc describes: *"Per-invoice: predicted payment date
range (P10, P50, P90) + confidence flag"*. Wrapping the whole pipeline in one function makes it
trivial to call from a FastAPI endpoint later — you pass in raw invoice rows, get back dates.

In [14]:
def predict_payment_window(raw_df, preprocessor, rf_model, sector_priors, global_prior):
    """
    raw_df must contain: issue_date, sector, invoice_amount, payment_term_days,
    customer_avg_payment_days, customer_recent_avg_payment_days, previous_payment_days,
    customer_payment_std, payment_behavior_trend, customer_invoice_count
    (i.e. the same feature-engineering output as notebook 01's model1_features.csv)
    """
    frame = raw_df.copy()
    frame["is_cold_start"] = frame["customer_invoice_count"] < MIN_HISTORY_FOR_CONFIDENCE

    sector_fill = frame["sector"].map(sector_priors).fillna(global_prior)
    for col in ["customer_avg_payment_days", "customer_recent_avg_payment_days", "previous_payment_days"]:
        frame[col] = frame[col].fillna(sector_fill)
    frame["customer_payment_std"] = frame["customer_payment_std"].fillna(0)
    frame["payment_behavior_trend"] = frame["payment_behavior_trend"].fillna(0)

    X = preprocessor.transform(frame[feature_columns])
    preds = rf_quantile_predict(rf_model, X)  # already p10 <= p50 <= p90 by construction

    out = frame[["issue_date"]].copy()
    for id_col in ["invoice_id", "cust_number"]:
        if id_col in frame.columns:
            out.insert(0, id_col, frame[id_col])

    out["predicted_days_p10"] = preds["p10"].values
    out["predicted_days_p50"] = preds["p50"].values
    out["predicted_days_p90"] = preds["p90"].values

    out["predicted_pay_date_p10"] = frame["issue_date"] + pd.to_timedelta(preds["p10"].values, unit="D")
    out["predicted_pay_date_p50"] = frame["issue_date"] + pd.to_timedelta(preds["p50"].values, unit="D")
    out["predicted_pay_date_p90"] = frame["issue_date"] + pd.to_timedelta(preds["p90"].values, unit="D")

    out["confidence"] = np.where(frame["is_cold_start"], "low", "normal")
    return out


validation_output = predict_payment_window(
    validation_df, preprocessor, rf_quantile_model, sector_priors, global_prior
)
validation_output.head(10)

,cust_number,invoice_id,issue_date,predicted_days_p10,predicted_days_p50,predicted_days_p90,predicted_pay_date_p10,predicted_pay_date_p50,predicted_pay_date_p90,confidence
30,C1000,INV700031,2025-12-04,59.0,67.0,77.0,2026-02-01 00:00:00,2026-02-09,2026-02-19 00:00:00,normal
31,C1000,INV700032,2026-01-02,64.0,71.0,78.0,2026-03-07 00:00:00,2026-03-14,2026-03-21 00:00:00,normal
32,C1000,INV700033,2026-02-14,59.9,69.0,75.0,2026-04-14 21:36:00,2026-04-24,2026-04-30 00:00:00,normal
63,C1001,INV700070,2025-10-07,85.0,91.0,96.0,2025-12-31 00:00:00,2026-01-06,2026-01-11 00:00:00,normal
64,C1001,INV700071,2025-10-17,89.0,90.0,101.0,2026-01-14 00:00:00,2026-01-15,2026-01-26 00:00:00,normal
65,C1001,INV700072,2025-11-15,85.0,89.0,97.1,2026-02-08 00:00:00,2026-02-12,2026-02-20 02:24:00,normal
66,C1001,INV700073,2025-11-20,83.0,88.0,97.1,2026-02-11 00:00:00,2026-02-16,2026-02-25 02:24:00,normal
67,C1001,INV700074,2025-12-06,88.0,90.0,101.0,2026-03-04 00:00:00,2026-03-06,2026-03-17 00:00:00,normal
68,C1001,INV700075,2025-12-14,86.9,91.0,98.0,2026-03-10 21:36:00,2026-03-15,2026-03-22 00:00:00,normal
69,C1001,INV700076,2025-12-16,84.0,89.0,98.0,2026-03-10 00:00:00,2026-03-15,2026-03-24 00:00:00,normal


In [15]:
output_dir = Path("../data/processed")
validation_output.to_csv(output_dir / "model1_validation_predictions.csv", index=False)
print(f"saved {len(validation_output)} rows to {output_dir / 'model1_validation_predictions.csv'}")

saved 745 rows to ..\data\processed\model1_validation_predictions.csv


## 11. Build features for OPEN invoices (not just the closed/validation set)

Everything above (`train_df`/`validation_df`/`test_df`) only ever covered `status == "closed"`
invoices — that's correct for training and evaluation, since you need a known `days_to_payment`
to train or score against. But your teammate's Monte Carlo engine needs predictions for
**currently open, unpaid invoices** — the real outstanding cash. Those were filtered out before
any feature engineering happened, so we build their features here: for each open invoice, use
that customer's payment history *as of now* (all their closed invoices), exactly the same
features `historical_df` computed, just evaluated once per customer rather than expanding
invoice-by-invoice.

Note: `disputed_open` invoices are included as one candidate set below, but flagged separately -
worth confirming with your teammate whether disputed invoices should go through the same
days-to-payment model at all, or be handled as a separate risk category (a 2-year-old dispute is
a different kind of problem than a normal late payment).

In [16]:
raw_df = pd.read_csv("../data/raw/invoices.csv")
raw_df["issue_date"] = pd.to_datetime(raw_df["issue_date"])

closed_history = raw_df[raw_df["status"] == "closed"].copy()
open_invoices = raw_df[raw_df["status"].isin(["open", "disputed_open"])].copy()

print(f"open: {(open_invoices['status'] == 'open').sum()}, "
      f"disputed_open: {(open_invoices['status'] == 'disputed_open').sum()}")

open: 264, disputed_open: 83


In [17]:
# Per-customer stats from ALL of that customer's closed history (as of "now" - the whole
# closed set is in the past relative to any open invoice in this dataset, so no leakage).
closed_history_sorted = closed_history.sort_values(["cust_number", "issue_date"])

customer_latest_stats = closed_history_sorted.groupby("cust_number").agg(
    customer_avg_payment_days=("days_to_payment", "mean"),
    customer_recent_avg_payment_days=("days_to_payment", lambda x: x.tail(3).mean()),
    customer_payment_std=("days_to_payment", "std"),
    previous_payment_days=("days_to_payment", "last"),
    customer_invoice_count=("days_to_payment", "count"),
)
customer_latest_stats["payment_behavior_trend"] = (
    customer_latest_stats["customer_recent_avg_payment_days"]
    - customer_latest_stats["customer_avg_payment_days"]
)

open_features_df = open_invoices.merge(customer_latest_stats, on="cust_number", how="left")

# genuinely new customers (no closed history at all) get NaN here -> caught by is_cold_start
# + sector-prior fallback inside predict_payment_window, same as during training
open_features_df[list(customer_latest_stats.columns)].isna().sum()

customer_avg_payment_days           0
customer_recent_avg_payment_days    0
customer_payment_std                0
previous_payment_days               0
customer_invoice_count              0
payment_behavior_trend              0
dtype: int64

## 12. Run open invoices through the saved Model 1 pipeline

In [18]:
open_predictions = predict_payment_window(
    open_features_df, preprocessor, rf_quantile_model, sector_priors, global_prior
)

# round to whole days before this goes anywhere - fractional days produce nonsense timestamps
for col in ["predicted_days_p10", "predicted_days_p50", "predicted_days_p90"]:
    open_predictions[col] = np.round(open_predictions[col]).astype(int)

open_predictions.head(10)

,cust_number,invoice_id,issue_date,predicted_days_p10,predicted_days_p50,predicted_days_p90,predicted_pay_date_p10,predicted_pay_date_p50,predicted_pay_date_p90,confidence
0,C1000,INV700035,2026-03-20,59,65,78,2026-05-18 00:00:00,2026-05-24 00:00:00,2026-06-06,normal
1,C1000,INV700040,2026-06-20,59,65,78,2026-08-18 00:00:00,2026-08-24 00:00:00,2026-09-06,normal
2,C1000,INV700041,2026-07-06,59,67,76,2026-09-03 00:00:00,2026-09-11 00:00:00,2026-09-20,normal
3,C1000,INV700042,2026-08-04,59,65,78,2026-10-02 00:00:00,2026-10-08 00:00:00,2026-10-21,normal
4,C1000,INV700043,2026-08-05,59,67,78,2026-10-03 00:00:00,2026-10-11 00:00:00,2026-10-22,normal
5,C1000,INV700044,2026-08-09,59,67,78,2026-10-07 00:00:00,2026-10-15 00:00:00,2026-10-26,normal
6,C1001,INV700082,2026-03-08,84,91,97,2026-05-31 00:00:00,2026-06-07 00:00:00,2026-06-13,normal
7,C1002,INV700122,2026-05-07,30,38,50,2026-06-05 21:36:00,2026-06-14 00:00:00,2026-06-26,normal
8,C1002,INV700127,2026-08-01,29,40,50,2026-08-29 21:36:00,2026-09-10 12:00:00,2026-09-20,normal
9,C1002,INV700128,2026-08-03,29,40,50,2026-09-01 00:00:00,2026-09-12 00:00:00,2026-09-22,normal


In [19]:
open_predictions["confidence"].value_counts()

confidence
normal    347
Name: count, dtype: int64

## 13. Export as the JSON payload for Model 2

Matches the exact shape your teammate asked for. `customer_id` maps from `cust_number`.

In [20]:
import json

def to_model2_payload(prediction_df):
    return [
        {
            "invoice_id": row["invoice_id"],
            "customer_id": row["cust_number"],
            "p10_payment_days": int(row["predicted_days_p10"]),
            "p50_payment_days": int(row["predicted_days_p50"]),
            "p90_payment_days": int(row["predicted_days_p90"]),
        }
        for _, row in prediction_df.iterrows()
    ]

model2_payload = to_model2_payload(open_predictions)

output_path = Path("../data/processed/model1_to_model2_payload.json")
with open(output_path, "w") as f:
    json.dump(model2_payload, f, indent=2)

print(f"saved {len(model2_payload)} open-invoice predictions to {output_path}")
model2_payload[:3]

saved 347 open-invoice predictions to ..\data\processed\model1_to_model2_payload.json


[{'invoice_id': 'INV700035',
  'customer_id': 'C1000',
  'p10_payment_days': 59,
  'p50_payment_days': 65,
  'p90_payment_days': 78},
 {'invoice_id': 'INV700040',
  'customer_id': 'C1000',
  'p10_payment_days': 59,
  'p50_payment_days': 65,
  'p90_payment_days': 78},
 {'invoice_id': 'INV700041',
  'customer_id': 'C1000',
  'p10_payment_days': 59,
  'p50_payment_days': 67,
  'p90_payment_days': 76}]

## Where this leaves you vs. the spec

| Spec requirement | Status |
|---|---|
| Days-to-payment regressor | Done (notebook 02 - RF 7.27 MAE / XGB 7.89 MAE, kept as reference) |
| P10/P50/P90 quantile output | Done (this notebook) |
| Cold-start fallback to sector-average prior | Done |
| Low-confidence flag | Done |
| Evaluation (pinball loss + P10-P90 coverage) | Done |
| Saved, reusable model artifacts | Done |
| Per-invoice output feeding Model 2 / SHAP | Done |
| Open-invoice feature build + JSON handoff for Model 2 | Done |

Next up per your build-priority table: **Model 2 (Monte Carlo)** consumes
`model1_to_model2_payload.json` directly — each invoice's P10/P50/P90 becomes the distribution
your teammate samples from in the simulation. **Model 5 (SHAP)** runs `shap.TreeExplainer`
directly on `rf_quantile_model` — near-zero extra work once this notebook is saved (SHAP explains
the underlying RF's predictions; since the quantiles come from the same forest, the ranked
feature list is still meaningful context for why an interval came out wide or narrow).